# Notebook 3: Teacher Model Training

**What this does (in simple terms):**

This is where the "smart" model (teacher) learns to tell real audio
from fake audio. It uses two parts:

- **Wav2Vec2-XLSR** (front-end): A huge pretrained model that "listens"
  to raw audio and extracts meaningful patterns — like how your brain
  processes sound into recognizable features.

- **AASIST** (back-end): A graph attention network that looks at those
  patterns from both the frequency angle (pitch, tone) and time angle
  (rhythm, pauses) to find spoofing clues. Then it decides: real or fake.

The teacher is the big, powerful model. Later in Notebook 4, it will
teach a smaller, faster model (student) what it learned.

**Checkpoint strategy:** The model is saved after every epoch to Google
Drive. If Colab disconnects, you can restart and it will pick up
exactly where it left off — no lost training.

**Run on Colab Pro with GPU. Training takes ~8-15 hours.**

## 3.1 Setup

In [ ]:
# Mount Google Drive to save checkpoints persistently
# This ensures training progress survives Colab disconnections
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install HuggingFace transformers library for pretrained wav2vec2 models
# The -q flag suppresses verbose installation output
!pip install transformers -q

In [ ]:
# Import required libraries
import os, json, random, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model  # Pretrained speech model
from sklearn.metrics import accuracy_score, roc_curve
from tqdm import tqdm  # Progress bars
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")  # Suppress non-critical warnings

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
PREPROCESSED_DIR = Path("/content/preprocessed")  # Location of preprocessed audio
CHECKPOINT_DIR = Path("/content/drive/MyDrive/deepfake_project/checkpoints")  # Save to Drive
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)  # Create if doesn't exist

# Audio configuration
TARGET_SR = 16000  # 16kHz sampling rate (standard for speech models)
MAX_DURATION = 6  # Process 6-second audio chunks
MAX_SAMPLES = TARGET_SR * MAX_DURATION  # 96,000 samples per utterance

# Training hyperparameters
BATCH_SIZE = 8  # Process 8 utterances simultaneously (limited by GPU memory)
NUM_EPOCHS = 10  # Number of complete passes through training data

# Learning rates: differential learning for different model parts
LEARNING_RATE_W2V = 1e-5  # Low LR for pretrained wav2vec2 (fine-tuning)
LEARNING_RATE_BACKEND = 1e-4  # Higher LR for AASIST (training from scratch)

WEIGHT_DECAY = 1e-4  # L2 regularization to prevent overfitting

# Reproducibility: fix random seeds for consistent results
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # Also seed GPU operations

# Device selection: use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Sanity check: verify preprocessed data exists before training
assert PREPROCESSED_DIR.exists(), "Preprocessed data not found! Run Notebook 1 first."

# Count files in each split
for split in ["train", "dev"]:
    count = len(list((PREPROCESSED_DIR / split).glob("*.flac")))
    print(f"{split}: {count} files")

## 3.2 Load Protocols & Build Dataset

The ASVspoof protocol files contain metadata about each audio sample:
- speaker_id: Which person is speaking
- audio_id: Unique identifier for this utterance
- system_id: Which synthesis system was used (for spoofs)
- attack_type: Type of spoofing attack (A01-A19)
- label: 'bonafide' or 'spoof'

Same Dataset code from Notebook 2, included here so this
notebook is self-contained.

In [ ]:
def load_protocol(path):
    """Load ASVspoof protocol file.
    
    Format: speaker_id audio_id system_id attack_type label
    Example: LA_0079 LA_T_5138576 - - bonafide
    """
    df = pd.read_csv(
        path, 
        sep=" ",  # Space-separated values
        header=None,  # No column names in file
        names=["speaker_id", "audio_id", "system_id", "attack_type", "label"]
    )
    # Convert labels to integers: 0 = bonafide, 1 = spoof
    df["label_int"] = (df["label"] == "spoof").astype(int)
    return df

# Load train and dev protocols
PROTOCOL_DIR = PREPROCESSED_DIR / "protocols"
train_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.train.trn.txt")
dev_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.dev.trl.txt")

# Load metadata (original lengths from preprocessing)
metadata = pd.read_csv(PREPROCESSED_DIR / "metadata.csv")

print(f"Train: {len(train_protocol)} | Dev: {len(dev_protocol)}")

In [ ]:
# ============================================================
# RAWBOOST AUGMENTATION FUNCTIONS
# ============================================================
# RawBoost adds realistic noise to prevent overfitting to clean audio
# Paper: "RawBoost: A Raw Data Boosting and Augmentation Method" (ICASSP 2022)

def rawboost_convolutive(audio_np, sr=TARGET_SR):
    """Apply frequency-domain convolutive noise.
    
    Simulates real-world filtering effects from:
    - Codec compression (MP3, AAC)
    - Transmission channels (phone lines)
    - Recording equipment response
    
    Method: Create random notch filters in frequency domain
    """
    n = len(audio_np)
    spectrum = np.fft.rfft(audio_np)  # Convert to frequency domain
    freqs = np.fft.rfftfreq(n, d=1.0/sr)  # Frequency bins
    
    # Apply 1-4 random notch filters
    for _ in range(np.random.randint(1, 5)):
        f = np.random.uniform(100, sr//2-100)  # Random center frequency
        bw = np.random.uniform(50, 300)  # Random bandwidth
        # Gaussian notch centered at f with bandwidth bw
        spectrum *= 1.0 - np.exp(-0.5*((freqs-f)/(bw/2+1e-8))**2)
    
    return np.fft.irfft(spectrum, n=n).astype(np.float32)  # Back to time domain

def rawboost_impulsive(audio_np):
    """Add impulsive noise.
    
    Simulates:
    - Microphone artifacts
    - Electrical interference
    - Environmental clicks/pops
    
    Method: Add signal-dependent Gaussian noise
    """
    nl = np.random.uniform(0.001, 0.01)  # Noise level (0.1% - 1% of signal)
    # Noise is proportional to signal amplitude (louder parts get more noise)
    return (audio_np + nl*np.random.randn(len(audio_np))*np.abs(audio_np)).astype(np.float32)

def rawboost_augment(a):
    """Apply RawBoost augmentation (combination of convolutive + impulsive)."""
    a = a.copy()  # Don't modify original
    if np.random.rand() < 0.5:  # 50% chance
        a = rawboost_convolutive(a)
    if np.random.rand() < 0.5:  # 50% chance (can apply both)
        a = rawboost_impulsive(a)
    return a

def codec_augment(a, sr=TARGET_SR):
    """Simulate codec compression artifacts.
    
    Two methods:
    1. Mu-law encoding: 8-bit logarithmic quantization (used in telephony)
    2. Downsampling: 16kHz -> 8kHz -> 16kHz (simulates low-quality transmission)
    """
    t = torch.from_numpy(a).unsqueeze(0).float()  # Convert to tensor
    
    if np.random.rand() < 0.5:
        # Mu-law compression (256 quantization levels)
        t = torchaudio.functional.mu_law_decoding(
            torchaudio.functional.mu_law_encoding(t, 256), 256)
    else:
        # Downsample to 8kHz and back up (loses high-frequency detail)
        t = torchaudio.transforms.Resample(sr, 8000)(t)
        t = torchaudio.transforms.Resample(8000, sr)(t)
    
    return t.squeeze(0).numpy()

def apply_augmentation(audio_tensor):
    """Master augmentation function applied during training.
    
    Probabilities:
    - 70% chance of RawBoost (convolutive + impulsive)
    - 30% chance of codec simulation
    - Can apply both (21% chance)
    """
    a = audio_tensor.numpy().copy()
    if np.random.rand() < 0.7:
        a = rawboost_augment(a)
    if np.random.rand() < 0.3:
        a = codec_augment(a)
    return torch.from_numpy(a).float()

In [ ]:
# ============================================================
# DATASET CLASS
# ============================================================

class ASVspoofDataset(Dataset):
    """PyTorch Dataset for ASVspoof audio samples.
    
    Responsibilities:
    1. Load audio from disk
    2. Apply augmentation (training only)
    3. Normalize amplitude
    4. Create attention masks (distinguish real audio from padding)
    5. Return dictionary with audio, mask, label, metadata
    """
    
    def __init__(self, protocol_df, audio_dir, metadata_df, is_train=False):
        """
        Args:
            protocol_df: DataFrame with audio_id, label, attack_type, etc.
            audio_dir: Directory containing .flac files
            metadata_df: DataFrame with original_length for each audio
            is_train: If True, apply augmentation
        """
        self.protocol = protocol_df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.is_train = is_train
        
        # Build lookup table: audio_id -> original_length
        # Used to create proper attention masks (separate real audio from padding)
        self.length_lookup = {}
        for _, row in metadata_df.iterrows():
            if row.get("status") == "success":
                self.length_lookup[row["audio_id"]] = int(row["original_length"])

    def __len__(self):
        return len(self.protocol)

    def __getitem__(self, idx):
        """Load and process a single audio sample.
        
        Steps:
        1. Load audio from .flac file
        2. Apply augmentation if training
        3. Normalize amplitude to [-1, 1]
        4. Create attention mask
        5. Return dictionary
        """
        row = self.protocol.iloc[idx]
        
        # Load audio
        audio, _ = torchaudio.load(str(self.audio_dir / f"{row['audio_id']}.flac"))
        audio = audio.squeeze(0)  # Remove channel dimension: [1, N] -> [N]
        
        # Apply augmentation during training only
        if self.is_train:
            audio = apply_augmentation(audio)
        
        # Normalize amplitude: scale to [-1, 1] range
        # Prevents different recording levels from affecting the model
        mx = torch.max(torch.abs(audio))
        if mx > 0:
            audio = audio / mx
        
        # Create attention mask: 1 for real audio, 0 for padding
        # This tells wav2vec2 which parts to attend to
        orig_len = self.length_lookup.get(row["audio_id"], MAX_SAMPLES)
        mask = torch.zeros(MAX_SAMPLES)
        mask[:orig_len] = 1.0  # Mark first orig_len samples as real audio
        
        return {
            "audio": audio,  # [96000] waveform
            "mask": mask,  # [96000] attention mask
            "label": torch.tensor(row["label_int"], dtype=torch.long),  # 0 or 1
            "audio_id": row["audio_id"],  # For debugging
            "attack_type": row["attack_type"]  # A01-A19 or '-'
        }

In [ ]:
# Create dataset instances
train_dataset = ASVspoofDataset(
    train_protocol, 
    PREPROCESSED_DIR/"train", 
    metadata, 
    is_train=True  # Enable augmentation
)
dev_dataset = ASVspoofDataset(
    dev_protocol, 
    PREPROCESSED_DIR/"dev", 
    metadata, 
    is_train=False  # No augmentation during evaluation
)

# Create data loaders: handle batching, shuffling, parallel loading
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,  # Shuffle training data each epoch
    num_workers=2,  # Parallel data loading (CPU threads)
    pin_memory=True  # Faster GPU transfer
)
dev_loader = DataLoader(
    dev_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,  # Don't shuffle validation data
    num_workers=2, 
    pin_memory=True
)

print(f"Train: {len(train_loader)} batches | Dev: {len(dev_loader)} batches")

## 3.3 Model Architecture

The teacher model consists of:
1. **Wav2Vec2-XLSR**: Pretrained speech encoder (300M parameters)
2. **AASIST Backend**: Graph attention network for spoofing detection

Architecture follows the paper:
- Jung et al., "AASIST: Audio Anti-Spoofing using Integrated Spectro-Temporal Graph Attention Networks" (ICASSP 2022)

In [ ]:
# ============================================================
# GRAPH ATTENTION LAYER
# ============================================================

class GraphAttentionLayer(nn.Module):
    """Single graph attention layer.
    
    Graph attention allows each node to attend to other nodes based on learned importance.
    Unlike CNNs (local) or RNNs (sequential), graph attention models arbitrary relationships.
    
    Key idea: For node i, compute weighted sum of all nodes j where weights α_ij are learned.
    
    Steps:
    1. Linear projection: h = W·x
    2. Compute attention scores: α_ij = softmax(LeakyReLU(a^T [h_i || h_j]))
    3. Aggregate: h'_i = Σ_j α_ij · h_j
    """
    
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        # Linear transformation for node features
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        
        # Attention mechanism: learn importance of edges
        self.a_src = nn.Linear(out_dim, 1, bias=False)  # Source node contribution
        self.a_dst = nn.Linear(out_dim, 1, bias=False)  # Destination node contribution
        
        self.leaky_relu = nn.LeakyReLU(0.2)  # Non-linearity for attention scores
        self.dropout = nn.Dropout(dropout)  # Regularization
    
    def forward(self, x):
        """Forward pass.
        
        Args:
            x: [batch, num_nodes, in_dim] - graph node features
        
        Returns:
            [batch, num_nodes, out_dim] - updated node features after attention
        """
        # Step 1: Linear projection
        h = self.W(x)  # [batch, num_nodes, out_dim]
        
        # Step 2: Compute pairwise attention scores
        # Broadcasting: [batch, nodes, 1] + [batch, 1, nodes] -> [batch, nodes, nodes]
        attn = self.leaky_relu(
            self.a_src(h) +  # [batch, nodes, 1] - how much each node attends OUT
            self.a_dst(h).transpose(-2, -1)  # [batch, 1, nodes] - how much each node is attended TO
        )
        # attn[i,j] = attention score from node i to node j
        
        # Step 3: Apply softmax + dropout + weighted aggregation
        # softmax normalizes attention scores per source node (each row sums to 1)
        attn_weights = self.dropout(F.softmax(attn, dim=-1))  # [batch, nodes, nodes]
        
        # Matrix multiplication: weighted sum of neighbor features
        return torch.bmm(attn_weights, h)  # [batch, nodes, out_dim]


class MultiHeadGraphAttention(nn.Module):
    """Multi-head graph attention (like multi-head self-attention in Transformers).
    
    Multiple attention heads capture different types of relationships:
    - Head 1 might focus on harmonic relationships (frequency domain)
    - Head 2 might focus on temporal continuity (time domain)
    
    Outputs are concatenated and normalized.
    """
    
    def __init__(self, in_dim, out_dim, n_heads=2, dropout=0.1):
        super().__init__()
        # Create n_heads independent attention layers
        self.heads = nn.ModuleList([
            GraphAttentionLayer(in_dim, out_dim, dropout) 
            for _ in range(n_heads)
        ])
        # Layer normalization for stable training
        self.norm = nn.LayerNorm(out_dim * n_heads)  # Concatenated output size
    
    def forward(self, x):
        """Forward pass.
        
        Args:
            x: [batch, nodes, in_dim]
        
        Returns:
            [batch, nodes, out_dim * n_heads] - concatenated multi-head output
        """
        # Apply each head independently and concatenate
        # If out_dim=160, n_heads=2 -> output is [batch, nodes, 320]
        return self.norm(torch.cat([h(x) for h in self.heads], dim=-1))


# ============================================================
# AASIST BACKEND
# ============================================================

class AASSISTBackend(nn.Module):
    """AASIST backend: Dual-branch graph attention for spoofing detection.
    
    Key innovation: Process speech features through TWO independent graph attention branches:
    1. Spectral branch: Models frequency-domain relationships (harmonics, formants)
    2. Temporal branch: Models time-domain relationships (prosody, rhythm)
    
    Deepfake artifacts appear in different ways:
    - TTS: Often has overly smooth spectra, unnatural formants
    - VC: Often has temporal discontinuities, prosody issues
    
    By processing both independently then fusing, we catch artifacts in either domain.
    """
    
    def __init__(self, input_dim=1024, hidden_dim=160, n_heads=2, dropout=0.1):
        """
        Args:
            input_dim: Wav2vec2 output dimension (1024 for XLS-R)
            hidden_dim: Internal feature dimension (160 for teacher, 64 for student)
            n_heads: Number of attention heads (2 is standard)
            dropout: Dropout probability for regularization
        """
        super().__init__()
        
        # Input projection: 1024 -> hidden_dim
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),  # Smooth activation (better than ReLU for transformers)
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout)
        )
        
        # Spectral branch: graph attention over frequency-domain features
        self.spectral_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.spectral_pool = nn.AdaptiveAvgPool1d(1)  # Pool across nodes -> single vector
        
        # Temporal branch: graph attention over time-domain features  
        self.temporal_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)  # Pool across nodes -> single vector
        
        # After pooling: spectral=[batch, hidden*n_heads], temporal=[batch, hidden*n_heads]
        # Concatenation dimension
        cd = hidden_dim * n_heads * 2  # e.g., 160*2*2 = 640
        
        # Fusion layer: combine spectral and temporal insights
        self.hetero_attention = nn.Sequential(
            nn.Linear(cd, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Classification head: hidden_dim -> 2 logits (real, fake)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)  # Output: [logit_real, logit_fake]
        )

    def forward(self, features):
        """Forward pass through AASIST.
        
        Args:
            features: [batch, ~300 frames, 1024] from wav2vec2
        
        Returns:
            [batch, 2] logits for real/fake classification
        """
        # Project to hidden dimension
        x = self.projection(features)  # [batch, frames, hidden_dim]
        
        # Spectral branch: treat frames as graph nodes
        # Graph attention models relationships between different time frames
        spectral_out = self.spectral_gat(x)  # [batch, frames, hidden*n_heads]
        # Pool across frames: [batch, hidden*n_heads, frames] -> [batch, hidden*n_heads, 1]
        sp = self.spectral_pool(spectral_out.transpose(1, 2)).squeeze(-1)
        
        # Temporal branch: same architecture, different learned weights
        temporal_out = self.temporal_gat(x)  # [batch, frames, hidden*n_heads]
        tp = self.temporal_pool(temporal_out.transpose(1, 2)).squeeze(-1)
        
        # Concatenate spectral and temporal representations
        fused = torch.cat([sp, tp], dim=-1)  # [batch, cd]
        
        # Fusion + classification
        fused = self.hetero_attention(fused)  # [batch, hidden_dim]
        return self.classifier(fused)  # [batch, 2]


# ============================================================
# TEACHER MODEL
# ============================================================

class TeacherModel(nn.Module):
    """Complete teacher model: Wav2Vec2-XLSR + AASIST.
    
    Pipeline:
    1. Raw audio [batch, 96000] -> Wav2Vec2-XLSR frontend
    2. XLS-R features [batch, ~300, 1024] -> AASIST backend  
    3. AASIST logits [batch, 2] -> Classification
    
    Fine-tuning strategy:
    - CNN feature extractor: FROZEN (already perfect at acoustic features)
    - Transformer layers: FINE-TUNED (adapt to spoofing detection)
    - AASIST backend: TRAINED from scratch
    """
    
    def __init__(self, w2v_name="facebook/wav2vec2-xls-r-300m", hidden_dim=160):
        """
        Args:
            w2v_name: HuggingFace model name for wav2vec2
            hidden_dim: AASIST hidden dimension (160 for teacher)
        """
        super().__init__()
        
        # Load pretrained wav2vec2 from HuggingFace (takes 1-2 minutes)
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(w2v_name)
        
        # CRITICAL: Freeze CNN feature extractor
        # Only fine-tune transformer layers, keep CNN frozen
        self.wav2vec2.feature_extractor._freeze_parameters()
        
        # Initialize AASIST backend (random weights)
        self.backend = AASSISTBackend(
            input_dim=self.wav2vec2.config.hidden_size,  # 1024 for XLS-R
            hidden_dim=hidden_dim
        )

    def forward(self, audio, mask=None):
        """Forward pass.
        
        Args:
            audio: [batch, 96000] raw waveform
            mask: [batch, 96000] attention mask (1=real audio, 0=padding)
        
        Returns:
            [batch, 2] logits for classification
        """
        # Convert mask to long tensor if provided
        if mask is not None:
            mask = mask.long()
        
        # Wav2Vec2 forward pass
        # Output: last_hidden_state = [batch, ~300 frames, 1024]
        # The mask tells wav2vec2 which time steps are real audio vs padding
        features = self.wav2vec2(audio, attention_mask=mask).last_hidden_state
        
        # AASIST forward pass
        return self.backend(features)

    def get_param_groups(self):
        """Get parameter groups for differential learning rates.
        
        Returns:
            List of dicts with 'params' and 'lr' keys for optimizer
            
        Why different learning rates?
        - Wav2vec2 is pretrained: use LOW lr (1e-5) to fine-tune gently
        - AASIST is random: use HIGH lr (1e-4) to train from scratch
        
        Note: self.wav2vec2.parameters() includes both CNN (frozen) and
        transformer (trainable), but PyTorch optimizer automatically skips
        parameters with requires_grad=False.
        """
        return [
            {"params": self.wav2vec2.parameters(), "lr": LEARNING_RATE_W2V},
            {"params": self.backend.parameters(), "lr": LEARNING_RATE_BACKEND},
        ]

In [ ]:
def count_params(model, trainable_only=True):
    """Count model parameters.
    
    Args:
        model: PyTorch model
        trainable_only: If True, count only parameters with requires_grad=True
    
    Returns:
        Number of parameters
    """
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())

# Initialize teacher model
print("Loading Wav2Vec2-XLSR (takes 1-2 min)...")
teacher = TeacherModel().to(DEVICE)

# Print parameter counts
print(f"Total params:     {count_params(teacher, False):,}")
print(f"Trainable params: {count_params(teacher, True):,}")
# Difference = frozen CNN parameters

# Quick forward pass test to verify dimensions
with torch.no_grad():
    dummy_audio = torch.randn(2, MAX_SAMPLES).to(DEVICE)  # Batch of 2
    dummy_mask = torch.ones(2, MAX_SAMPLES).to(DEVICE)
    out = teacher(dummy_audio, dummy_mask)
    print(f"Output shape: {out.shape}")  # Should be [2, 2]

print("Model ready!")

## 3.4 Loss Function & Metrics

**OC-Softmax (One-Class Softmax):**
- Standard softmax treats all classes equally
- OC-Softmax adds class-specific margins to push classes apart
- Enforces tighter clustering in embedding space

**Equal Error Rate (EER):**
- Standard metric for spoofing detection
- Point where FAR (false accept rate) = FRR (false reject rate)
- Lower is better (0% = perfect, 50% = random)

In [ ]:
# ============================================================
# OC-SOFTMAX LOSS
# ============================================================

class OCSoftmax(nn.Module):
    """One-Class Softmax loss with angular margin.
    
    Paper: "Replay and Synthetic Speech Detection with Res2Net Architecture" (ICASSP 2021)
    
    Key idea: Add class-specific margins to logits before cross-entropy.
    
    Standard softmax: loss = -log(exp(z_y) / Σ exp(z_i))
    OC-Softmax:       loss = -log(exp(z_y - m_y) / Σ exp(z_i - m_i))
    
    Where m_y is the margin for the true class.
    
    Effect: Makes it harder to correctly classify (requires higher confidence).
    This forces the model to create more separated clusters in embedding space.
    
    Margin interpretation:
    - m_real = 0.9 (large): Real speech must form tight cluster
    - m_fake = 0.2 (small): Fake speech allowed to be more spread
    
    Why asymmetric margins?
    - Real speech is consistent (all follow same acoustic physics)
    - Fake speech is diverse (TTS, VC, neural vocoders all different)
    """
    
    def __init__(self, m_real=0.9, m_fake=0.2, alpha=20.0):
        """
        Args:
            m_real: Margin for bonafide class (larger = tighter cluster)
            m_fake: Margin for spoof class (smaller = looser cluster)
            alpha: Scaling factor (increases gradient magnitude)
        """
        super().__init__()
        self.m_real = m_real
        self.m_fake = m_fake
        self.alpha = alpha

    def forward(self, logits, labels):
        """Compute OC-Softmax loss.
        
        Args:
            logits: [batch, 2] raw model outputs
            labels: [batch] ground truth (0=real, 1=fake)
        
        Returns:
            Scalar loss value
        """
        # Create binary masks for each class
        real_mask = (labels == 0).float()  # [batch] - 1 if real, 0 if fake
        spoof_mask = (labels == 1).float()  # [batch] - 1 if fake, 0 if real
        
        # Subtract margins from logits
        mod = logits.clone()
        # For real samples: subtract m_real from real logit
        mod[:, 0] = logits[:, 0] - self.m_real * real_mask
        # For fake samples: subtract m_fake from fake logit  
        mod[:, 1] = logits[:, 1] - self.m_fake * spoof_mask
        
        # Scale and compute cross-entropy
        # alpha=20 amplifies gradients for faster training
        return F.cross_entropy(self.alpha * mod, labels)


# ============================================================
# EVALUATION METRICS
# ============================================================

def compute_eer(labels, scores):
    """Compute Equal Error Rate.
    
    EER is the operating point where:
    - FAR (False Accept Rate) = FRR (False Reject Rate)
    - FAR: % of fakes incorrectly accepted as real
    - FRR: % of reals incorrectly rejected as fake
    
    Args:
        labels: [N] ground truth (0=real, 1=fake)
        scores: [N] predicted probabilities of being fake
    
    Returns:
        eer: Equal error rate (scalar between 0 and 1)
        threshold: Optimal threshold at EER point
    """
    # Compute ROC curve: (FPR, TPR) pairs at different thresholds
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    
    # FNR = 1 - TPR (false negative rate)
    fnr = 1 - tpr
    
    # Find point where |FPR - FNR| is minimized
    # This is where FAR ≈ FRR
    idx = np.nanargmin(np.abs(fpr - fnr))
    
    # EER is average of FPR and FNR at this point
    eer = (fpr[idx] + fnr[idx]) / 2
    
    return eer, thresholds[idx]


@torch.no_grad()  # Disable gradient computation for evaluation
def evaluate(model, dataloader, criterion, device):
    """Evaluate model on validation/test set.
    
    Args:
        model: PyTorch model
        dataloader: DataLoader for eval data
        criterion: Loss function
        device: cuda or cpu
    
    Returns:
        Dict with 'loss', 'eer', 'accuracy'
    """
    model.eval()  # Set to evaluation mode (disables dropout, etc.)
    
    # Accumulators
    total_loss = 0
    all_scores = []  # Probability of being fake
    all_preds = []   # Hard predictions (0 or 1)
    all_labels = []  # Ground truth
    
    for batch in tqdm(dataloader, desc="Evaluating"):
        # Move batch to device
        audio = batch["audio"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["label"].to(device)
        
        # Forward pass
        logits = model(audio, mask)
        
        # Compute loss
        total_loss += criterion(logits, labels).item()
        
        # Extract predictions
        # Softmax converts logits to probabilities: [logit_real, logit_fake] -> [P(real), P(fake)]
        probs = F.softmax(logits, dim=-1)
        all_scores.extend(probs[:, 1].cpu().numpy())  # P(fake)
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())  # 0 or 1
        all_labels.extend(labels.cpu().numpy())
    
    # Convert to numpy arrays
    labels_arr = np.array(all_labels)
    scores_arr = np.array(all_scores)
    preds_arr = np.array(all_preds)
    
    # Compute metrics
    eer, _ = compute_eer(labels_arr, scores_arr)
    
    return {
        "loss": total_loss / len(dataloader),
        "eer": eer,
        "accuracy": accuracy_score(labels_arr, preds_arr)
    }

## 3.5 Training Loop with Checkpoint Resume

**Checkpoint strategy:**
- After every epoch, save full training state to Google Drive
- If Colab disconnects, restart notebook and training resumes automatically
- Two checkpoint files:
  - `teacher_latest.pt`: Complete state (model, optimizer, scheduler, history)
  - `teacher_best.pt`: Best model only (for deployment)

**Training workflow:**
1. Forward pass: audio -> model -> logits
2. Compute loss: OC-Softmax(logits, labels)
3. Backward pass: loss.backward() computes gradients
4. Gradient clipping: prevent exploding gradients
5. Optimizer step: update weights
6. Validation: evaluate on dev set
7. Save checkpoint

In [ ]:
# ============================================================
# INITIALIZE TRAINING COMPONENTS
# ============================================================

# Loss function
criterion = OCSoftmax().to(DEVICE)

# Optimizer: AdamW (Adam with weight decay = L2 regularization)
# Uses differential learning rates via get_param_groups()
optimizer = torch.optim.AdamW(
    teacher.get_param_groups(),  # Returns list of param groups with different LRs
    weight_decay=WEIGHT_DECAY  # L2 penalty to prevent overfitting
)

# Learning rate scheduler: Cosine annealing
# LR follows cosine curve from initial -> 0 over T_max epochs
# This gradual decay helps fine-tuning converge smoothly
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=NUM_EPOCHS  # Complete cycle over all epochs
)

In [ ]:
# ============================================================
# CHECKPOINT RESUME LOGIC
# ============================================================

TEACHER_CKPT_PATH = CHECKPOINT_DIR / "teacher_latest.pt"  # Full state for resume
TEACHER_BEST_PATH = CHECKPOINT_DIR / "teacher_best.pt"    # Best model for deployment

# Initialize training state
start_epoch = 1
best_dev_eer = float("inf")  # Track best validation EER
history = {
    "train_loss": [],
    "train_acc": [],
    "dev_eer": [],
    "dev_loss": []
}

# Check if checkpoint exists (from previous interrupted training)
if TEACHER_CKPT_PATH.exists():
    print("Found existing checkpoint! Resuming training...")
    
    # Load checkpoint
    ckpt = torch.load(str(TEACHER_CKPT_PATH), map_location=DEVICE)
    
    # Restore model weights
    teacher.load_state_dict(ckpt["model_state_dict"])
    
    # Restore optimizer state (momentum, learning rates, etc.)
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    
    # Restore scheduler state (current LR)
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    
    # Restore training progress
    start_epoch = ckpt["epoch"] + 1  # Continue from next epoch
    best_dev_eer = ckpt["best_dev_eer"]
    history = ckpt["history"]
    
    print(f"Resuming from epoch {start_epoch} (best EER so far: {best_dev_eer*100:.2f}%)")
else:
    print("No checkpoint found. Starting fresh training.")

In [ ]:
# ============================================================
# MAIN TRAINING LOOP
# ============================================================

print("=" * 60)
print(f"TEACHER TRAINING: Epochs {start_epoch} to {NUM_EPOCHS}")
print("=" * 60)

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f"\n{'='*40} Epoch {epoch}/{NUM_EPOCHS} {'='*40}")

    # ========================================
    # TRAINING PHASE
    # ========================================
    teacher.train()  # Enable training mode (dropout active, etc.)
    epoch_loss = 0
    epoch_preds = []
    epoch_labels = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
    for batch in pbar:
        # Move batch to GPU
        audio = batch["audio"].to(DEVICE)   # [batch, 96000]
        mask = batch["mask"].to(DEVICE)     # [batch, 96000]
        labels = batch["label"].to(DEVICE)  # [batch]

        # Zero gradients from previous iteration
        optimizer.zero_grad()
        
        # Forward pass
        logits = teacher(audio, mask)  # [batch, 2]
        
        # Compute loss
        loss = criterion(logits, labels)
        
        # Backward pass: compute gradients
        loss.backward()
        
        # Gradient clipping: prevent exploding gradients
        # Limits gradient norm to max_norm=1.0
        # Important for transformer models which can have unstable gradients
        torch.nn.utils.clip_grad_norm_(teacher.parameters(), max_norm=1.0)
        
        # Update weights
        # For each trainable parameter θ:
        #   θ_new = θ_old - lr × gradient
        # Different LRs for wav2vec2 (1e-5) vs AASIST (1e-4)
        optimizer.step()

        # Accumulate metrics for this epoch
        epoch_loss += loss.item()
        epoch_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        epoch_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # Compute epoch-level training metrics
    train_loss = epoch_loss / len(train_loader)
    train_acc = accuracy_score(epoch_labels, epoch_preds)

    # ========================================
    # VALIDATION PHASE
    # ========================================
    dev_res = evaluate(teacher, dev_loader, criterion, DEVICE)
    
    # Step learning rate scheduler
    # Cosine annealing: gradually reduce LR toward 0
    scheduler.step()

    # ========================================
    # LOGGING
    # ========================================
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["dev_eer"].append(dev_res["eer"])
    history["dev_loss"].append(dev_res["loss"])

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Dev Loss:   {dev_res['loss']:.4f} | Dev EER: {dev_res['eer']*100:.2f}%")

    # ========================================
    # SAVE BEST MODEL
    # ========================================
    # Save model if it achieves new best validation EER
    if dev_res["eer"] < best_dev_eer:
        best_dev_eer = dev_res["eer"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": teacher.state_dict(),
                "dev_eer": best_dev_eer
            },
            str(TEACHER_BEST_PATH)
        )
        print(f"  ★ New best model saved! (EER: {best_dev_eer*100:.2f}%)")

    # ========================================
    # SAVE CHECKPOINT FOR RESUME
    # ========================================
    # Save complete training state after EVERY epoch
    # This allows seamless resume if Colab disconnects
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_dev_eer": best_dev_eer,
            "history": history,
        },
        str(TEACHER_CKPT_PATH)
    )
    print(f"  Checkpoint saved to Google Drive (epoch {epoch})")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE! Best Dev EER: {best_dev_eer*100:.2f}%")
print(f"{'='*60}")

## 3.6 Training Plots

Visualize training progress:
- **Loss**: Should decrease over epochs (both train and dev)
- **Training Accuracy**: Should increase (approaching 100%)
- **Dev EER**: Should decrease (lower is better)

Warning signs:
- Train loss decreasing but dev loss increasing = overfitting
- Both losses plateauing early = underfitting or learning rate too low
- Dev EER not improving = model not learning useful patterns

In [ ]:
# Create 3-panel plot showing training progress
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Loss curves (train vs dev)
axes[0].plot(history["train_loss"], "o-", label="Train", linewidth=2)
axes[0].plot(history["dev_loss"], "o-", label="Dev", linewidth=2)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("Loss Curves", fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Panel 2: Training accuracy
axes[1].plot(history["train_acc"], "o-", color="green", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Accuracy", fontsize=12)
axes[1].set_title("Training Accuracy", fontsize=14)
axes[1].grid(True, alpha=0.3)

# Panel 3: Validation EER (main metric)
axes[2].plot([e*100 for e in history["dev_eer"]], "o-", color="red", linewidth=2)
axes[2].set_xlabel("Epoch", fontsize=12)
axes[2].set_ylabel("EER (%)", fontsize=12)
axes[2].set_title("Dev EER (Lower is Better)", fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot to Google Drive
plt.savefig(
    str(CHECKPOINT_DIR / "teacher_training_plots.png"), 
    dpi=150, 
    bbox_inches="tight"
)
plt.show()

print("\n✅ Notebook 3 complete. Proceed to Notebook 4: Knowledge Distillation.")